<a href="https://colab.research.google.com/github/Chi123Zhang/Finance-Research/blob/main/LLM%20reason.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [205]:
# =========================
# 0) Setup: clone repo
# =========================
REPO_URL = "https://github.com/Chi123Zhang/Finance-Research.git"
REPO_DIR = "Finance-Research"

from pathlib import Path
import os

if not Path(REPO_DIR).exists():
    !git clone {REPO_URL}

%cd {REPO_DIR}
!ls -lh

Cloning into 'Finance-Research'...
remote: Enumerating objects: 110, done.
remote: Counting objects: 100% (110/110), done.
remote: Compressing objects: 100% (89/89), done.
remote: Total 110 (delta 24), reused 88 (delta 15), pack-reused 0 (from 0)
Receiving objects: 100% (110/110), 1.43 MiB | 24.35 MiB/s, done.
Resolving deltas: 100% (24/24), done.
/content/Finance-Research/Finance-Research/Finance-Research/Finance-Research
total 28K
drwxr-xr-x 2 root root 4.0K Mar  5 15:26 app
drwxr-xr-x 5 root root 4.0K Mar  5 15:26 chunk
drwxr-xr-x 7 root root 4.0K Mar  5 15:26 data
drwxr-xr-x 2 root root 4.0K Mar  5 15:26 llm
-rw-r--r-- 1 root root   18 Mar  5 15:26 README.md
-rw-r--r-- 1 root root  909 Mar  5 15:26 run_all.py
drwxr-xr-x 3 root root 4.0K Mar  5 15:26 strategy


In [206]:
# =========================
# 1) Load monthly_chunks + feature matrix
# =========================
import pandas as pd
import numpy as np

CHUNKS_PATH = Path("chunk/sample/monthly_chunks.parquet")
X_PATH      = Path("chunk/sample/monthly_feat_matrix.npy")

assert CHUNKS_PATH.exists(), f"Missing: {CHUNKS_PATH}"
assert X_PATH.exists(), f"Missing: {X_PATH}"

monthly_chunks = pd.read_parquet(CHUNKS_PATH)
X = np.load(X_PATH)

print("monthly_chunks:", monthly_chunks.shape)
print("X:", X.shape)
monthly_chunks.head()

monthly_chunks: (420, 30)
X: (420, 17)


,symbol,month,start_date,end_date,n_days,curve_logret,curve_cum,feat_month_ret,feat_month_vol,feat_month_mdd,...,ctx_unemployment_change_mean,ctx_unemployment_change_end,ctx_unemployment_change_chg,ctx_pe_ratio_end,ctx_ps_ratio_end,ctx_rev_growth_qoq_end,ctx_volume_change_end,label_next_month_ret,label_next_month_mdd,feat_vec
0,AAPL,2021-07,2021-07-03,2021-07-31,29,"[0.0, 0.0, 0.0, 0.0146112252544602, 0.01779592...","[1.0, 1.0, 1.0, 1.0, 1.0147184909974278, 1.032...",0.042155,0.178651,-0.044921,...,0.0,0.0,0.0,6.172662e-09,1.628193e-09,0.000000,-1.000000,0.040930,-0.031498,"[0.22477113097898765, -0.5335889694717059, 0.5..."
1,AAPL,2021-08,2021-08-01,2021-08-31,31,"[0.0, -0.0023337233462202, 0.0125650382971298,...","[1.0, 1.0, 0.9976689976689976, 1.0102838338132...",0.040930,0.151949,-0.031498,...,0.0,0.0,0.0,6.425307e-09,1.694834e-09,0.000000,0.708916,-0.068037,-0.096943,"[0.21599984617952556, -0.6411139230448345, 0.7..."
2,AAPL,2021-09,2021-09-01,2021-09-30,30,"[0.0044686937739958, 0.0074471209084012, 0.004...","[1.0, 1.0044786932753735, 1.0119870908252648, ...",-0.068037,0.171650,-0.096943,...,0.0,0.0,0.0,6.507542e-09,1.737603e-09,-0.090976,0.443733,0.058657,-0.024606,"[-0.5640694132485541, -0.5617820137476452, 0.0..."
3,AAPL,2021-10,2021-10-01,2021-10-31,31,"[0.0080943605762167, 0.0, 0.0, -0.024913457164...","[1.0, 1.0081272084805655, 1.0081272084805655, ...",0.058657,0.152249,-0.024606,...,0.0,0.0,0.0,6.889257e-09,1.839526e-09,-0.090976,-1.000000,0.103471,-0.031678,"[0.3429083217380599, -0.6399068945017538, 0.78..."
4,AAPL,2021-11,2021-11-01,2021-11-30,30,"[-0.0056232575543622, 0.0070908050127334, 0.00...","[1.0, 0.9943925233644859, 1.0014686248331108, ...",0.103471,0.192010,-0.031678,...,0.0,0.0,0.0,7.602097e-09,2.029865e-09,-0.090976,1.926668,0.074229,-0.054054,"[0.6637237543031062, -0.47979520314802615, 0.7..."


In [207]:
# =========================
# 2) Build cosine KNN index
# =========================
from sklearn.neighbors import NearestNeighbors

# cosine distance: smaller is more similar; we will convert to similarity = 1 - dist
nn = NearestNeighbors(n_neighbors=50, metric="cosine")
nn.fit(X)

print("✅ KNN index built.")

✅ KNN index built.


In [208]:
# =========================
# 3) Query function: find Top-K similar months + build evidence + build compare table
# =========================
import numpy as np

def query_similar(symbol: str, month: str, k: int = 10, exclude_self: bool = True):
    """
    symbol: e.g. "AAPL"
    month:  e.g. "2024-11"
    returns:
      res: DataFrame with top-k similar rows
      evidence: dict contains query/top1 meta + curves
    """
    mask = (monthly_chunks["symbol"] == symbol) & (monthly_chunks["month"] == month)
    if mask.sum() == 0:
        raise ValueError(f"Cannot find chunk for {symbol} {month}. Check available months.")

    q_idx = int(np.where(mask.to_numpy())[0][0])
    q_vec = X[q_idx].reshape(1, -1)

    # get more neighbors then filter self
    dist, idx = nn.kneighbors(q_vec, n_neighbors=max(k + 5, 20))
    idx = idx.flatten()
    dist = dist.flatten()
    sim = 1 - dist  # cosine similarity

    out = []
    for i, s in zip(idx, sim):
        if exclude_self and i == q_idx:
            continue
        out.append((i, float(s)))
        if len(out) >= k:
            break

    # build result table
    rows = []
    for rank, (i, s) in enumerate(out, start=1):
        r = monthly_chunks.iloc[i]
        rows.append({
            "rank": rank,
            "similarity": s,
            "symbol": r.get("symbol"),
            "month": r.get("month"),
            "start_date": r.get("start_date"),
            "end_date": r.get("end_date"),
            # some common feature cols (if exist)
            "feat_month_ret": r.get("feat_month_ret", np.nan),
            "feat_month_vol": r.get("feat_month_vol", np.nan),
            "feat_month_mdd": r.get("feat_month_mdd", np.nan),
            "feat_trend_slope": r.get("feat_trend_slope", np.nan),
            "label_next_month_ret": r.get("label_next_month_ret", np.nan),
        })
    res = pd.DataFrame(rows)

    # evidence curves (used for plotting / debugging)
    q_row = monthly_chunks.iloc[q_idx]
    top1_i = out[0][0]
    top1_row = monthly_chunks.iloc[top1_i]

    q_curve = np.array(q_row["curve_cum"], dtype=float)
    t_curve = np.array(top1_row["curve_cum"], dtype=float)

    # normalize start=1
    q_curve = q_curve / (q_curve[0] if q_curve[0] != 0 else 1.0)
    t_curve = t_curve / (t_curve[0] if t_curve[0] != 0 else 1.0)

    evidence = {
        "query": {"symbol": symbol, "month": month},
        "top1": {"symbol": top1_row["symbol"], "month": top1_row["month"], "similarity": out[0][1]},
        "q_curve": q_curve,
        "top1_curve": t_curve,
        "q_idx": q_idx,
        "top1_idx": int(top1_i),
    }
    return res, evidence


def build_compare_table(monthly_chunks: pd.DataFrame, evidence: dict) -> pd.DataFrame:
    """
    Create the Query vs Top1 feature compare table like your screenshot.
    index: feature name
    cols:  Query, Top1
    """
    q_idx = evidence["q_idx"]
    t_idx = evidence["top1_idx"]
    q_row = monthly_chunks.iloc[q_idx]
    t_row = monthly_chunks.iloc[t_idx]

    # choose columns: ctx_* plus selected feat_*
    wanted = [c for c in monthly_chunks.columns if c.startswith("ctx_")]
    wanted += [
        "feat_month_ret", "feat_month_vol", "feat_month_mdd", "feat_trend_slope",
        "feat_ret_first_half", "feat_ret_second_half", "feat_vol20_chg",
    ]
    # keep only existing
    wanted = [c for c in wanted if c in monthly_chunks.columns]

    compare = pd.DataFrame({
        "Query": [q_row[c] for c in wanted],
        "Top1":  [t_row[c] for c in wanted],
    }, index=wanted)

    # make sure numeric where possible
    compare["Query"] = pd.to_numeric(compare["Query"], errors="coerce")
    compare["Top1"] = pd.to_numeric(compare["Top1"], errors="coerce")

    return compare

In [209]:
# =========================
# 4) Pick which (symbol, month) you want -> generate compare.csv automatically
# =========================
QUERY_SYMBOL = "AAPL"
QUERY_MONTH  = "2024-11"
K = 10

res, evidence = query_similar(QUERY_SYMBOL, QUERY_MONTH, k=K)
display(res)

compare = build_compare_table(monthly_chunks, evidence)
display(compare)

# save compare.csv into repo
OUT_COMPARE = Path("chunk/sample/compare.csv")
compare.to_csv(OUT_COMPARE)
print("✅ saved:", OUT_COMPARE, " shape:", compare.shape)

,rank,similarity,symbol,month,start_date,end_date,feat_month_ret,feat_month_vol,feat_month_mdd,feat_trend_slope,label_next_month_ret
0,1,0.915144,MSFT,2024-11,2024-11-01,2024-11-30,0.042107,0.149969,-0.032842,0.416760,-0.004629
1,2,0.773160,TSM,2023-03,2023-03-01,2023-03-31,0.068336,0.226279,-0.045470,0.672630,-0.093743
2,3,0.772982,GOOGL,2023-12,2023-12-01,2023-12-31,0.054026,0.200612,-0.036442,0.841226,0.002935
3,4,0.733556,GOOGL,2024-06,2024-06-01,2024-06-30,0.055942,0.140193,-0.017583,0.880094,-0.058249
4,5,0.721462,QCOM,2023-12,2023-12-01,2023-12-31,0.120728,0.146859,-0.019479,0.935257,0.026827
5,6,0.718059,TXN,2023-03,2023-03-01,2023-03-31,0.084923,0.178343,-0.025990,0.786893,-0.101124
6,7,0.717160,MSFT,2023-12,2023-12-01,2023-12-31,-0.007574,0.114457,-0.034256,0.396680,0.057281
7,8,0.710325,AAPL,2023-03,2023-03-01,2023-03-31,0.118649,0.187231,-0.034649,0.907027,0.028987
8,9,0.696421,GOOGL,2023-07,2023-07-01,2023-07-31,0.108772,0.259220,-0.049593,0.712301,0.025995
9,10,0.694582,MSFT,2023-03,2023-03-01,2023-03-31,0.155882,0.221252,-0.032234,0.889773,0.065765


,Query,Top1
ctx_real_rate_mean,5.236616e-01,5.236616e-01
ctx_real_rate_end,5.236616e-01,5.236616e-01
ctx_real_rate_chg,0.000000e+00,0.000000e+00
ctx_yield_curve_mean,-2.800000e-01,-2.800000e-01
ctx_yield_curve_end,-2.800000e-01,-2.800000e-01
ctx_yield_curve_chg,0.000000e+00,0.000000e+00
ctx_unemployment_change_mean,-7.936508e-04,-7.936508e-04
ctx_unemployment_change_end,0.000000e+00,0.000000e+00
ctx_unemployment_change_chg,2.380952e-02,2.380952e-02
ctx_pe_ratio_end,1.106537e-08,1.921674e-08


✅ saved: chunk/sample/compare.csv  shape: (20, 2)


In [210]:
print(compare.columns)

Index(['Query', 'Top1'], dtype='object')


In [211]:
print(res.columns)
print(res.head())

Index(['rank', 'similarity', 'symbol', 'month', 'start_date', 'end_date',
       'feat_month_ret', 'feat_month_vol', 'feat_month_mdd',
       'feat_trend_slope', 'label_next_month_ret'],
      dtype='object')
   rank  similarity symbol    month start_date   end_date  feat_month_ret  \
0     1    0.915144   MSFT  2024-11 2024-11-01 2024-11-30        0.042107   
1     2    0.773160    TSM  2023-03 2023-03-01 2023-03-31        0.068336   
2     3    0.772982  GOOGL  2023-12 2023-12-01 2023-12-31        0.054026   
3     4    0.733556  GOOGL  2024-06 2024-06-01 2024-06-30        0.055942   
4     5    0.721462   QCOM  2023-12 2023-12-01 2023-12-31        0.120728   

   feat_month_vol  feat_month_mdd  feat_trend_slope  label_next_month_ret  
0        0.149969       -0.032842          0.416760             -0.004629  
1        0.226279       -0.045470          0.672630             -0.093743  
2        0.200612       -0.036442          0.841226              0.002935  
3        0.140193       

In [212]:
import pandas as pd

# 1) Segment 2：严格用 query_symbol + query_month 在 monthly_chunks 里定位
q = monthly_chunks[(monthly_chunks["symbol"] == QUERY_SYMBOL) & (monthly_chunks["month"] == QUERY_MONTH)]
if q.empty:
    raise ValueError(f"Query not found in monthly_chunks: {QUERY_SYMBOL} {QUERY_MONTH}")

qrow = q.iloc[0]
seg2 = {
    "symbol": QUERY_SYMBOL,
    "start": pd.to_datetime(qrow["start_date"]),
    "end": pd.to_datetime(qrow["end_date"]),
}

# 2) Segment 1：从 res 里挑“历史最相似”，排除同 symbol & 同 month（避免拿自己/同月）
seg1_row = None
for _, row in res.iterrows():
    if (row["symbol"] == QUERY_SYMBOL) and (row["month"] == QUERY_MONTH):
        continue
    # 你也可以只排除同 month：if row["month"] == QUERY_MONTH: continue
    seg1_row = row
    break

if seg1_row is None:
    raise ValueError("No suitable historical segment found in res.")

seg1 = {
    "symbol": seg1_row["symbol"],
    "start": pd.to_datetime(seg1_row["start_date"]),
    "end": pd.to_datetime(seg1_row["end_date"]),
}

print("Segment 1:", seg1)
print("Segment 2:", seg2)

Segment 1: {'symbol': 'MSFT', 'start': Timestamp('2024-11-01 00:00:00'), 'end': Timestamp('2024-11-30 00:00:00')}
Segment 2: {'symbol': 'AAPL', 'start': Timestamp('2024-11-01 00:00:00'), 'end': Timestamp('2024-11-30 00:00:00')}


In [213]:
# =========================
# 5) Build Segments (two companies, same month)
# =========================

segments = [
    {
        "segment_id": 1,
        "symbol": seg1["symbol"],
        "start_date": seg1["start"].strftime("%Y-%m-%d"),
        "end_date": seg1["end"].strftime("%Y-%m-%d"),
    },
    {
        "segment_id": 2,
        "symbol": seg2["symbol"],
        "start_date": seg2["start"].strftime("%Y-%m-%d"),
        "end_date": seg2["end"].strftime("%Y-%m-%d"),
    },
]

print("Segments:\n", segments)


print(f"""
Segment 1:
{seg1['symbol']} {seg1['start'].strftime('%Y-%m-%d')} → {seg1['end'].strftime('%Y-%m-%d')}

Segment 2:
{seg2['symbol']} {seg2['start'].strftime('%Y-%m-%d')} → {seg2['end'].strftime('%Y-%m-%d')}
""")

Segments:
 [{'segment_id': 1, 'symbol': 'MSFT', 'start_date': '2024-11-01', 'end_date': '2024-11-30'}, {'segment_id': 2, 'symbol': 'AAPL', 'start_date': '2024-11-01', 'end_date': '2024-11-30'}]

Segment 1:
MSFT 2024-11-01 → 2024-11-30

Segment 2:
AAPL 2024-11-01 → 2024-11-30



In [214]:
def build_llm_table(seg1, seg2, compare: pd.DataFrame):

    rows = []

    r1 = {
        "segment_id": 1,
        "company": f"{seg1['symbol']} (Top1)",
        "start_date": seg1["start"].strftime("%Y-%m-%d"),
        "end_date": seg1["end"].strftime("%Y-%m-%d"),
    }

    r2 = {
        "segment_id": 2,
        "company": f"{seg2['symbol']} (Query)",
        "start_date": seg2["start"].strftime("%Y-%m-%d"),
        "end_date": seg2["end"].strftime("%Y-%m-%d"),
    }

    for feat in compare.index:
        r1[feat] = float(compare.loc[feat, "Top1"])
        r2[feat] = float(compare.loc[feat, "Query"])

    rows.append(r1)
    rows.append(r2)

    return pd.DataFrame(rows)
segment_df = build_llm_table(seg1, seg2, compare)
segment_df = build_llm_table(seg1, seg2, compare)

print(segment_df)

   segment_id       company  start_date    end_date  ctx_real_rate_mean  \
0           1   MSFT (Top1)  2024-11-01  2024-11-30            0.523662   
1           2  AAPL (Query)  2024-11-01  2024-11-30            0.523662   

   ctx_real_rate_end  ctx_real_rate_chg  ctx_yield_curve_mean  \
0           0.523662                0.0                 -0.28   
1           0.523662                0.0                 -0.28   

   ctx_yield_curve_end  ctx_yield_curve_chg  ...  ctx_ps_ratio_end  \
0                -0.28                  0.0  ...      6.542247e-09   
1                -0.28                  0.0  ...      2.766826e-09   

   ctx_rev_growth_qoq_end  ctx_volume_change_end  feat_month_ret  \
0                 0.04638                   -1.0        0.042107   
1                -0.05483                   -1.0        0.050551   

   feat_month_vol  feat_month_mdd  feat_trend_slope  feat_ret_first_half  \
0        0.149969       -0.032842          0.416760             0.021287   
1        0

In [215]:
segment_table_md = segment_df.to_markdown(index=False)
print(segment_table_md)

|   segment_id | company      | start_date   | end_date   |   ctx_real_rate_mean |   ctx_real_rate_end |   ctx_real_rate_chg |   ctx_yield_curve_mean |   ctx_yield_curve_end |   ctx_yield_curve_chg |   ctx_unemployment_change_mean |   ctx_unemployment_change_end |   ctx_unemployment_change_chg |   ctx_pe_ratio_end |   ctx_ps_ratio_end |   ctx_rev_growth_qoq_end |   ctx_volume_change_end |   feat_month_ret |   feat_month_vol |   feat_month_mdd |   feat_trend_slope |   feat_ret_first_half |   feat_ret_second_half |   feat_vol20_chg |
|-------------:|:-------------|:-------------|:-----------|---------------------:|--------------------:|--------------------:|-----------------------:|----------------------:|----------------------:|-------------------------------:|------------------------------:|------------------------------:|-------------------:|-------------------:|-------------------------:|------------------------:|-----------------:|-----------------:|-----------------:|--------------

In [216]:
from pathlib import Path

def build_market_prompt(segment_table_md: str) -> str:
    return f"""
You are a quantitative market strategist writing an institutional-style market commentary.

Write analysis for TWO market segments.

IMPORTANT RULES
Output plain text only. Bullet points using "-" are allowed.
Do not output reasoning steps.
Use only numerical values from the table.
Round numbers to 2 decimals.
Write numbers as decimals (do not convert to percentages).
Follow the structure exactly.

You may include qualitative macro interpretation such as:

- monetary policy conditions
- discount rate environment
- investor risk appetite
- technology sector sentiment
- institutional capital flows into large-cap technology stocks

Do not introduce numerical values that are not present in the table.

Segment mapping
Segment 1 = Top1 similar segment
Segment 2 = Query segment

STRUCTURE

Segment X: company, start_date to end_date

Paragraph 1 — Market Regime
Describe the macroeconomic regime and investor sentiment during the period.
Mention technology sector positioning and general market stability.

Paragraph 2 — Price Dynamics
Explain price behavior using feat_month_ret and the overall trend.

Paragraph 3 — Macro Conditions
Interpret the macro indicators:

Real interest rate ≈ ctx_real_rate_mean
Yield curve ≈ ctx_yield_curve_mean
Unemployment change ≈ ctx_unemployment_change_mean

Explain what these indicators imply about discount rates and economic stability.

Paragraph 4 — Equity Market Implications
Explain how this macro regime influences investor positioning,
risk appetite, and capital flows toward large-cap technology stocks.

Paragraph 5 — Market Statistics
Introduce the statistical profile, then list:

- Monthly return ≈ feat_month_ret
- Monthly volatility ≈ feat_month_vol
- Maximum drawdown ≈ feat_month_mdd
- Trend slope ≈ feat_trend_slope

Paragraph 6 — Risk-Return Interpretation
Explain the asset’s momentum strength, downside risk,
and overall risk-return profile.

IMPORTANT FOR SEGMENT 2

Do NOT repeat Segment 1 wording.
Highlight differences in market behavior, especially trend slope
(feat_trend_slope) and volatility.

FINAL PARAGRAPH (Segment 2 only)

Explain why Segment 2 is similar to Segment 1.

This explanation must be at least 60 words and must:

- compare macro indicators
  (ctx_real_rate_mean, ctx_yield_curve_mean, ctx_unemployment_change_mean)

- compare market indicators
  (feat_month_ret, feat_month_vol, feat_month_mdd, feat_trend_slope)

- explain the economic mechanism behind the similarity,
  such as shared discount-rate environments,
  synchronized capital flows,
  or comparable volatility regimes.

Write Segment 1 first, then Segment 2.

Data:
{segment_table_md}
""".strip()

prompt = build_market_prompt(segment_table_md)
Path("prompt.txt").write_text(prompt, encoding="utf-8")
print("prompt saved")

prompt saved


In [217]:
from huggingface_hub import hf_hub_download

repo_id = "unsloth/Qwen3.5-35B-A3B-GGUF"
filename = "Qwen3.5-35B-A3B-Q4_K_M.gguf"

local_path = hf_hub_download(
    repo_id=repo_id,
    filename=filename
)

In [218]:
# =========================
# 8) (Optional) Run local LLM with llama.cpp -> analysis.md
#    如果你还没有 gguf 模型，先跳过这一格
# =========================
from pathlib import Path

# Build llama.cpp if needed
if not Path("llama.cpp").exists():
    !git clone https://github.com/ggerganov/llama.cpp

%cd llama.cpp
!cmake -B build -DGGML_CUDA=ON
!cmake --build build -j
%cd ..

# >>> IMPORTANT: Set your local gguf model path here <<<
MODEL_PATH = "/root/.cache/huggingface/hub/models--unsloth--Qwen3.5-35B-A3B-GGUF/snapshots/0da7d49832a73abe50f0c89070971b59dad0039d/Qwen3.5-35B-A3B-Q4_K_M.gguf"

assert Path(MODEL_PATH).exists(), f"❌ MODEL_PATH not found: {MODEL_PATH}"



Cloning into 'llama.cpp'...
remote: Enumerating objects: 81823, done.
remote: Counting objects: 100% (128/128), done.
remote: Compressing objects: 100% (116/116), done.
remote: Total 81823 (delta 50), reused 14 (delta 12), pack-reused 81695 (from 3)
Receiving objects: 100% (81823/81823), 309.04 MiB | 42.47 MiB/s, done.
Resolving deltas: 100% (58828/58828), done.
/content/Finance-Research/Finance-Research/Finance-Research/Finance-Research/llama.cpp
-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
CMAKE_BUILD_TYPE=Release
-- Found Git: /usr

In [219]:
!./llama.cpp/build/bin/llama-cli \
  -m "{MODEL_PATH}" \
  -f prompt.txt \
  -c 8192 \
  -ngl 50 \
  --temp 0.0 \
  2>&1 | tee analysis.md

ggml_cuda_init: found 1 CUDA devices:
  Device 0: NVIDIA RTX PRO 6000 Blackwell Server Edition, compute capability 12.0, VMM: yes

Loading model... |-\|/-\|/-\|/-\|/-\|/-\ 


▄▄ ▄▄
██ ██
██ ██  ▀▀█▄ ███▄███▄  ▀▀█▄    ▄████ ████▄ ████▄
██ ██ ▄█▀██ ██ ██ ██ ▄█▀██    ██    ██ ██ ██ ██
██ ██ ▀█▄██ ██ ██ ██ ▀█▄██ ██ ▀████ ████▀ ████▀
                                    ██    ██
                                    ▀▀    ▀▀

build      : b8211-a0ed91a44
model      : Qwen3.5-35B-A3B-Q4_K_M.gguf
modalities : text

available commands:
  /exit or Ctrl+C     stop or exit
  /regen              regenerate the last response
  /clear              clear the chat history
  /read               add a text file


> You are a quantitative market strategist writing an institutional-style market commentary.

Write analysis for TWO market segments.

IMPORTANT RULES
Output plain text only. Bullet points using "-" are allowed.
Do not output reasoning steps.
Use only numerical values from 

In [222]:
from pathlib import Path

raw = Path("analysis.md").read_text(errors="ignore")


def extract_final_output(text: str) -> str:

    if "</think>" in text:
        text = text.split("</think>")[-1]

    if "<<<BEGIN_FINAL>>>" in text and "<<<END_FINAL>>>" in text:
        text = text.split("<<<BEGIN_FINAL>>>", 1)[1].split("<<<END_FINAL>>>", 1)[0]

    return text.strip()


final_text = extract_final_output(raw)

Path("final.md").write_text(final_text, encoding="utf-8")

print("saved final.md, chars =", len(final_text))

print("\n--- preview ---\n")
print("\n".join(final_text.splitlines()[:120]))

saved final.md, chars = 3703

--- preview ---

Segment 1: MSFT (Top1), 2024-11-01 to 2024-11-30

During the period from 2024-11-01 to 2024-11-30, the market operated within a stable macroeconomic regime characterized by cautious investor sentiment. Technology sector positioning remained constructive as large-cap names attracted steady institutional capital flows. General market stability was maintained despite external volatility pressures.

Price behavior exhibited a positive trajectory with a monthly return of 0.04. The asset demonstrated resilience, capturing upside momentum while managing downside exposure effectively throughout the month.

Macro indicators reflected a neutral discount rate environment with a real interest rate of 0.52. The yield curve stood at -0.28, suggesting mild inversion, while unemployment change remained near -0.00. These conditions imply stable economic fundamentals with limited pressure on valuation multiples.

This macro regime supports risk-on positioni

In [223]:
import os
print(os.getcwd())

/content/Finance-Research/Finance-Research/Finance-Research/Finance-Research
